In [61]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve

In [62]:
class TxnDataProcessor:
    def __init__(self, url):
        self.url = url
        self.scaler = StandardScaler()
        self.column_names = None

    def load_and_clean(self):
        # Loading from a public mirror of the UCI/Kaggle Credit Card dataset
        df = pd.read_csv(self.url)
        # print(df.sample(5, random_state=5))

        # Feature Engineering: Time is usually an offset, Amount needs scaling
        df['Amount'] = self.scaler.fit_transform(df['Amount'].values.reshape(-1, 1))
        df = df.drop(['Time'], axis=1)
        
        self.column_names = df.columns.tolist()
        return df

    def prepare_split(self, df):
        # Anomaly detection strategy: Train ONLY on Class 0/normal txns
        normal_df = df[df['Class'] == 0].drop('Class', axis=1)
        fraud_df = df[df['Class'] == 1].drop('Class', axis=1)

        # Split normal data for training and validation
        train_data, val_data = train_test_split(normal_df, test_size=0.15, random_state=42)
        
        # Create a test set with both Normal and Fraud for evaluation
        # We take a subset of normal data to keep the test set balanced for metrics
        test_normal = val_data.sample(len(fraud_df), random_state=42)
        X_test = pd.concat([test_normal, fraud_df])
        y_test = [0] * len(test_normal) + [1] * len(fraud_df)
        
        return train_data.values, val_data.values, X_test.values, np.array(y_test)

In [ ]:
# Path/URL to a mirror of the Credit Card Fraud dataset
data_path = "Data/creditcard.csv"  # Online Url or Local File Path

# Step 1: Data Processing
processor = TxnDataProcessor(data_path)
full_df = processor.load_and_clean()
X_train, X_val, X_test, y_test = processor.prepare_split(full_df)

In [64]:
# Custom callback function to avoid printing metrics for every training epoch (Optional/Can be omitted)

class PrintCall(tf.keras.callbacks.Callback):
    def __init__(self, interval=10):
        self.interval = interval
        
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.interval == 0:
            print(f"\nEpoch {epoch + 1}", end=" ")
            for k, v in logs.items():
                print(f"{k}: {v:.4f}", end=" ")

In [65]:
class AnomalyAutoencoder(Model):
    def __init__(self, input_dim):
        super(AnomalyAutoencoder, self).__init__()
        # Encoder: Compresses input into a "bottleneck"
        self.encoder = tf.keras.Sequential([
            layers.Input(shape=(input_dim,)),
            layers.Dense(16, activation="relu"),
            layers.Dense(8, activation="relu")
        ])
        # Decoder: Attempts to reconstruct the original input
        self.decoder = tf.keras.Sequential([
            layers.Dense(16, activation="relu"),
            layers.Dense(input_dim, activation="linear")
        ])
        self.threshold = None

    def call(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

    def train_model(self, train_data, val_data, epochs=30, batch_size=512):
        self.compile(optimizer='adam', loss='mae')
        history = self.fit(
            train_data, train_data,
            epochs=epochs,
            batch_size=batch_size,
            validation_data=(val_data, val_data),
            verbose=0,
            callbacks=[PrintCall(interval=5)]
        )
        # Threshold = MAE + 2*Std Deviations
        reconstructions = self.predict(train_data)
        train_loss = tf.keras.losses.mae(reconstructions, train_data)
        self.threshold = np.mean(train_loss) + (2 * np.std(train_loss))
        return history

    def evaluate_performance(self, X_test, y_test):
        from sklearn.metrics import classification_report, confusion_matrix
        
        reconstructions = self.predict(X_test)
        test_loss = tf.keras.losses.mae(reconstructions, X_test)
        
        # If loss > threshold, it's an anomaly (1), else normal (0)
        predictions = tf.cast(tf.math.greater(test_loss, self.threshold), tf.int32)
        
        print(f"\nModel Threshold: {self.threshold:.4f}")
        print("\nConfusion Matrix:")
        print(confusion_matrix(y_test, predictions))
        print("\nClassification Report:")
        print(classification_report(y_test, predictions))

In [66]:
# Step 2: Model Building & Training
model = AnomalyAutoencoder(input_dim=X_train.shape[1])
model.train_model(X_train, X_val, epochs=20)

# Step 3: Evaluation
model.evaluate_performance(X_test, y_test)


Epoch 5 loss: 0.3470 val_loss: 0.3442 
Epoch 10 loss: 0.3291 val_loss: 0.3292 
Epoch 15 loss: 0.3136 val_loss: 0.3140 
7553/7553 ━━━━━━━━━━━━━━━━━━━━ 33s 4ms/step
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step

Model Threshold: 0.7362

Confusion Matrix:
[[475  17]
 [ 67 425]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.97      0.92       492
           1       0.96      0.86      0.91       492

    accuracy                           0.91       984
   macro avg       0.92      0.91      0.91       984
weighted avg       0.92      0.91      0.91       984



In [67]:
# Step 4a: Single Inference Example (Normal Txn)
sample_tx = X_test[0:1]
reconstruction = model.predict(sample_tx)
error = tf.keras.losses.mae(reconstruction, sample_tx).numpy()[0]
status = "ANOMALY" if error > model.threshold else "NORMAL"
print(f"\nInference on sample transaction: {status} (Error: {error:.4f})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 262ms/step

Inference on sample transaction: NORMAL (Error: 0.1810)


In [68]:
# Stp 4b: Single Inference Example (Fraud/Anomalous Txn)
neg_sample = X_test[-2:-1]
print(neg_sample)
reconstruction = model.predict(neg_sample)
error = tf.keras.losses.mae(reconstruction, neg_sample).numpy()[0]
status = "ANOMALY" if error > model.threshold else "NORMAL"
print(f"\nInference on sample transaction: {status} (Error: {error:.4f})")

[[-3.11383161  0.58586417 -5.39973021  1.81709247 -0.84061847 -2.94354779
  -2.20800192  1.05873268 -1.63233335 -5.24598384  1.93351954 -5.0304648
  -1.12745458 -6.41662798  0.14123723 -2.54949824 -4.61471707 -1.47813794
  -0.03548037  0.30627074  0.583276   -0.26920864 -0.45610777 -0.18365913
  -0.32816776  0.60611581  0.88487554 -0.25370032  0.62630172]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step

Inference on sample transaction: ANOMALY (Error: 1.7180)


In [69]:
# Random Inference 
neg_sample = full_df[full_df['Class'] == 1].iloc[-28]
neg_sample = neg_sample.values[:-1].reshape((1,-1))
reconstruction = model.predict(neg_sample)
error = tf.keras.losses.mae(reconstruction, neg_sample).numpy()[0]
status = "ANOMALY" if error > model.threshold else "NORMAL"
print(f"\nInference on sample transaction: {status} (Error: {error:.4f})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step

Inference on sample transaction: ANOMALY (Error: 2.2649)
